# Post-submission survey analysis

The eligible sample contains 12 respondents. Results use question-specific valid denominators; missingness remains separate from code absence. Multi-select and code counts can sum above the valid-response denominator because one respondent can contribute to several categories. Fields omitted from processed/public inputs: Q1–Q2 (device/submission identifiers) and Q31–Q32 (follow-up contact items).

Run all cells in order. This notebook reads processed aggregates and writes two result tables under `outputs/`. Plots display inline. For even samples, the numerical median averages the two central scores; it can lie between Likert categories. The full category distributions are retained.

All post-submission tables and figure labels show counts and valid-response denominators only. The `percent` field in the derived results CSV is empty by design; numerical medians, ranges and coding-coverage summaries are retained.

Q14–Q16 describe the five respondents who reported Q-Submission use in Q13. The seven nonusers are excluded only from these questions; `sample_n` remains 12, while `excluded_n` is 7 and applicable/valid n are 5. All five users answered all three questions. Q14 has complete ratings among applicable users: the seven original nonratings came from nonusers. Original response states remain preserved.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DATA = ROOT / "data/processed"
OUTPUT = ROOT / "outputs"
OUTPUT.mkdir(parents=True, exist_ok=True)

SURVEY = "postsub"
# Results-review exclusions affect analysis only; the processed data retain these questions.
ANALYSIS_EXCLUDED = set()
counts = pd.read_csv(DATA / "survey_counts.csv", dtype={"category_id": str})
counts = counts.loc[counts.survey_id == SURVEY].copy()
assert ANALYSIS_EXCLUDED <= set(counts.question_id)
counts = counts.loc[~counts.question_id.isin(ANALYSIS_EXCLUDED)].copy()
coverage = pd.read_csv(DATA / "survey_code_coverage.csv")
coverage = coverage.loc[(coverage.survey_id == SURVEY) & ~coverage.question_id.isin(ANALYSIS_EXCLUDED)]
assert not counts.duplicated(["question_id", "category_id"]).any()
assert counts["sample_n"].eq(12).all()
assert counts.n.ge(0).all() and counts.n.le(counts.valid_n).all()
assert counts.applicable_n.eq(counts.sample_n-counts.skip_n-counts.not_applicable_n-counts.excluded_n).all()
assert counts.valid_n.eq(counts.applicable_n-counts.blank_n-counts.explicit_missing_n).all()
results, summaries = {}, {}


In [ ]:
%matplotlib inline
import textwrap
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.patches import Rectangle

# Scientific figure style v1.1, with count-scaled Likert stacks and small-sample bars.
try:
    font_manager.findfont("Arial", fallback_to_default=False)
    font = "Arial"
except ValueError:
    font = "DejaVu Sans"
    print("Arial unavailable; using DejaVu Sans.")
plt.rcdefaults()
plt.rcParams.update({"font.family": font, "font.size": 11, "figure.facecolor": "white",
    "axes.facecolor": "white", "axes.edgecolor": "#404040", "axes.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "axes.axisbelow": True,
    "grid.color": "#DDDDDD", "grid.linewidth": 0.6, "legend.fontsize": 10})
COLORS = ["#2E5661", "#89ACB8", "#D8D4C7", "#F2C48D", "#E6A052"]
DENOMINATORS = ["sample_n", "applicable_n", "valid_n", "blank_n", "explicit_missing_n", "skip_n", "not_applicable_n", "excluded_n"]

def show_table(table):
    display(table[DENOMINATORS].iloc[:1])
    display(table[["category_label", "n", "n/N"]].style)

def plot_question(table):
    valid = int(table.valid_n.iloc[0])
    kind = table.question_type.iloc[0]
    if valid < 2 or table.n.gt(0).sum() <= 1:
        print("Figure omitted: fewer than two valid responses or observed categories.")
        return
    t = table.loc[table.n > 0] if kind == "coded" else table
    labels = t.category_label.astype(str).tolist()
    heading = None
    if (SURVEY, int(t.question_id.iloc[0])) in [("postsub", 18), ("presub", 15)]:
        heading = "Number of changes"
        labels = [s.replace(" changes", "").replace(" change", "") for s in labels]
    fractions = [f"n = {n}/{valid}" for n in t.n]
    if kind == "numeric":
        values = np.repeat(t.numeric_value.to_numpy(), t.n.to_numpy())
        levels = pd.Series(values).groupby(values).cumcount()
        fig, ax = plt.subplots(figsize=(10, 3), layout="constrained")
        ax.scatter(values, levels, s=36, color="#5F828E", edgecolors="#404040", linewidths=0.6)
        ax.set(xlabel="Discussion duration (minutes)", xticks=np.unique(values), yticks=[], ylim=(-0.6, max(levels)+0.8))
        ax.spines["left"].set_visible(False)
    elif kind == "likert" and valid > 5:
        labels = [textwrap.fill(s, 38, break_long_words=False) for s in labels]
        fig, ax = plt.subplots(figsize=(10, 3 + 0.18*sum(s.count("\n")+1 for s in labels)), layout="constrained")
        left, bars, texts = 0, [], []
        for n, label, color, fraction in zip(t.n, labels, COLORS, fractions):
            bar = ax.barh(0, n, left=left, height=0.5, color=color, edgecolor="#404040", linewidth=0.6, label=label)[0]
            bars.append(bar)
            if n:
                texts.append((ax.text(left+n/2, 0, fraction, ha="center", va="center", fontsize=10,
                    color="white" if color == COLORS[0] else "#202020"), bar))
            left += n
        ax.add_patch(Rectangle((0, -0.25), valid, 0.5, fill=False, edgecolor="#404040", linewidth=0.6, clip_on=False))
        ax.set(xlim=(0, valid), xticks=range(valid+1), xlabel="Valid respondents (n)", yticks=[], ylim=(-0.6, 0.6))
        ax.spines["left"].set_visible(False)
        ax.xaxis.grid(True)
        ax.legend(handles=bars, labels=labels, loc="upper center", bbox_to_anchor=(0.5, -0.35), ncols=2,
            fancybox=False, framealpha=1, edgecolor="#404040")
        fig.canvas.draw()
        renderer = fig.canvas.get_renderer()
        for text, bar in texts:
            text.set_visible(text.get_window_extent(renderer).width+6 <= bar.get_window_extent(renderer).width)
    else:
        vertical = len(t) <= 5
        labels = [textwrap.fill(s, 20 if vertical else 44, break_long_words=False, break_on_hyphens=False) for s in labels]
        height = 4+0.18*max(s.count("\n") for s in labels) if vertical else max(3, 1+0.31*sum(s.count("\n")+1 for s in labels))
        fig, ax = plt.subplots(figsize=(10, height), layout="constrained")
        maximum = int(t.n.max())
        if vertical:
            bars = ax.bar(range(len(t)), t.n, width=0.65, color="#5F828E", edgecolor="#404040", linewidth=0.6)
            ax.set(xticks=range(len(t)), xticklabels=labels, ylabel="Respondents (n)", ylim=(0, maximum+0.8), yticks=range(maximum+1))
            if heading: ax.set_xlabel(heading)
            ax.yaxis.grid(True)
        else:
            bars = ax.barh(range(len(t)), t.n, height=0.65, color="#5F828E", edgecolor="#404040", linewidth=0.6)
            ax.set(yticks=range(len(t)), yticklabels=labels, xlabel="Respondents (n)", xlim=(0, maximum+0.8), xticks=range(maximum+1))
            if heading: ax.set_ylabel(heading)
            ax.invert_yaxis()
            ax.xaxis.grid(True)
            fractions = [s.replace("\n", " ") for s in fractions]
        ax.bar_label(bars, labels=fractions, padding=4, fontsize=10)
    display(fig)
    plt.close(fig)


## Q3. What was the primary rationale for including a PCCP in your marketing submission? (e.g., more efficient change management, on-market adaptation/self-learning, resource predictability, etc.)

In [ ]:
table = counts.loc[counts.question_id == 3].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[3] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[3] = summary
plot_question(table)


## Q4. How clear and useful was the FDA final guidance in supporting your PCCP preparation and submission?

In [ ]:
table = counts.loc[counts.question_id == 4].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[4] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[4] = summary
plot_question(table)


## Q5. Which elements of the guidance were especially helpful, and why? (e.g., terminology, examples, interpretive flexibility, structure, alignment with practice, transparency.)

In [ ]:
table = counts.loc[counts.question_id == 5].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[5] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[5] = summary
plot_question(table)


## Q6. Which elements of the guidance were difficult to interpret or apply, and why? (e.g., scope boundaries, risk-interpretation, justification requirements, process burden.)

In [ ]:
table = counts.loc[counts.question_id == 6].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[6] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[6] = summary
plot_question(table)


## Q7. Please rate the difficulty your team encountered when preparing PCCP section VI: Description of modification

Higher scores indicate greater preparation challenge.

In [ ]:
table = counts.loc[counts.question_id == 7].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[7] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[7] = summary
plot_question(table)


## Q8. Please describe any difficulty or success you had with navigating the Description of modification section.

In [ ]:
table = counts.loc[counts.question_id == 8].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[8] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[8] = summary
plot_question(table)


## Q9. Please rate the difficulty your team encountered when preparing PCCP section VII: Modification Protocol

Higher scores indicate greater preparation challenge.

In [ ]:
table = counts.loc[counts.question_id == 9].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[9] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[9] = summary
plot_question(table)


## Q10. Please describe any difficulty or success you had with navigating the Modification Protocol section.

In [ ]:
table = counts.loc[counts.question_id == 10].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[10] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[10] = summary
plot_question(table)


## Q11. Please rate the difficulty your team encountered when preparing PCCP section VIII: Impact Assessment

Higher scores indicate greater preparation challenge.
The median of 3.5 averages the sixth ordered rating (3) and seventh (4); it lies between “Moderately” and “Very”, rather than identifying an observed response category.


In [ ]:
table = counts.loc[counts.question_id == 11].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[11] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[11] = summary
plot_question(table)


## Q12. Please describe any difficulty or success you had with navigating the Impact Assessment section.

In [ ]:
table = counts.loc[counts.question_id == 12].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[12] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[12] = summary
plot_question(table)


## Q13. Have you used the FDA’s Q-Submission Program to get feedback on your proposed PCCP before submitting a marketing application?

In [ ]:
table = counts.loc[counts.question_id == 13].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[13] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[13] = summary
plot_question(table)


## Q14. How useful was the FDA’s feedback during the Q-Submission process for improving your PCCP?

Analysis population: the five confirmed Q-Submission users (Q13 = Yes). All five provided a rating; seven nonusers are excluded from the applicable population. Higher scores indicate greater perceived usefulness.


In [ ]:
table = counts.loc[counts.question_id == 14].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[14] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[14] = summary
plot_question(table)


## Q15. Did FDA feedback during Q-Submission significantly alter your overall PCCP approach?
Analysis population: the five confirmed Q-Submission users (Q13 = Yes). Seven nonusers are excluded from this item, irrespective of whether they answered it. Q16 is not further restricted to respondents reporting a significant overall change in Q15.


In [ ]:
table = counts.loc[counts.question_id == 15].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[15] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[15] = summary
plot_question(table)


## Q16. Which aspects of your PCCP did FDA feedback lead you to revise? (select all that apply)
Analysis population: the five confirmed Q-Submission users (Q13 = Yes). Seven nonusers are excluded from this item, irrespective of whether they answered it. Q16 is not further restricted to respondents reporting a significant overall change in Q15.

The single Other selection is specified as “Other: types and number of modifications” using that respondent’s Q17 explanation. It remains one selection by one of the five confirmed users, and total selections remain eight.


In [ ]:
table = counts.loc[counts.question_id == 16].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[16] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summary["total_selections"] = int(table.n.sum())
print("Total selections:", summary["total_selections"])
summaries[16] = summary
plot_question(table)


## Q17. Supporting detail integrated into Q16

The sole substantive answer clarifies the existing Other category in Q16. Its two original codes are retained in the supporting coded results, coverage and reliability records. No separate Q17 substantive result table or figure is presented.


In [ ]:
table = counts.loc[counts.question_id == 17].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
results[17] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[17] = summary


## Q18. How many distinct design modifications are included in your device’s PCCP? Select the total number of planned change types (e.g., model retraining; threshold tuning; preprocessing/postprocessing; architecture adjustments; new modalities; UI/workflow improvements; feature additions; performance optimization):

In [ ]:
table = counts.loc[counts.question_id == 18].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[18] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[18] = summary
plot_question(table)


## Q19. Indicate the current adaptiveness of the AI algorithm in your PCCP submission:

In [ ]:
table = counts.loc[counts.question_id == 19].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[19] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[19] = summary
plot_question(table)


## Q20. Indicate your preferred future adaptiveness of the AI algorithm (2–5 year outlook):

In [ ]:
table = counts.loc[counts.question_id == 20].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[20] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[20] = summary
plot_question(table)


## Q21. How would you characterize your experience using the PCCP concept on your own devices?

Higher scores indicate greater perceived effectiveness.

Scale: 1 = Not at all effective; 2 = Slightly effective; 3 = Moderately effective; 4 = Very effective; 5 = Highly effective. “Prefer not to answer” is an additional option, counted as explicit missingness and excluded from the valid-rating denominator.


In [ ]:
table = counts.loc[counts.question_id == 21].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[21] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[21] = summary
plot_question(table)


## Q22. To what extent did the PCCP concept meet your expectations for managing post-market device modifications (overall approach)?

Higher scores indicate greater perceived effectiveness.

Scale: 1 = Not at all effective; 2 = Slightly effective; 3 = Moderately effective; 4 = Very effective; 5 = Highly effective. “Prefer not to answer” is an additional option, counted as explicit missingness and excluded from the valid-rating denominator.

Ratings of respondents’ experience with the PCCP concept (Q21) and the extent to which it met expectations (Q22) had identical distributions: 2/11 selected Slightly effective, 6/11 Moderately effective and 3/11 Very effective for each item. These related appraisals are summarized together while retaining both question-specific tables and their original response labels.


In [ ]:
table = counts.loc[counts.question_id == 22].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[22] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[22] = summary
plot_question(table)


## Q23. If your future AI/ML-enabled device would be eligible for a PCCP, would you choose to use the PCCP pathway in future submissions?

In [ ]:
table = counts.loc[counts.question_id == 23].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[23] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[23] = summary
plot_question(table)


## Q24. Which aspects of the PCCP concept are you satisfied with, and why? (e.g., implementation practicality, evidence requirements, lifecycle flexibility, clarity of change boundaries, QMS alignment, FDA predictability, efficiency gains over supplements.)

In [ ]:
table = counts.loc[counts.question_id == 24].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[24] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[24] = summary
plot_question(table)


## Q25. Which aspects of the PCCP concept are you not satisfied with, and why? (e.g., complexity, evidence burden, ambiguous boundaries, limited flexibility, misalignment, unpredictability, documentation intensity.)

In [ ]:
table = counts.loc[counts.question_id == 25].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[25] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[25] = summary
plot_question(table)


## Q26. From a regulatory-professional standpoint (workload, process clarity/functionality), how does PCCP compare to traditional change processes (e.g., Special 510(k), PMA supplements)?

Higher scores indicate stronger preference for PCCP.

Scale: 1 = Strong preference for traditional processes; 2 = Some preference for traditional processes; 3 = Neutral; 4 = Some preference for PCCP; 5 = Strong preference for PCCP. “Prefer not to answer” is an additional option, counted as explicit missingness and excluded from the valid-rating denominator.


In [ ]:
table = counts.loc[counts.question_id == 26].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[26] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[26] = summary
plot_question(table)


## Q27. Please explain your reasoning and key factors that shaped your preference:

In [ ]:
table = counts.loc[counts.question_id == 27].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[27] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[27] = summary
plot_question(table)


## Q28. To what extent do you agree that the PCCP provides overall advantages—such as improved customer satisfaction and commercial success—that justify the effort required?

Higher scores indicate greater agreement and perceived advantage.

Scale: 1 = Strongly disagree (significantly less favorable); 2 = Disagree (less favorable); 3 = Neither more nor less favorable; 4 = Agree (more favorable); 5 = Strongly agree (significantly more favorable). “Prefer not to answer” is an additional option, counted as explicit missingness and excluded from the valid-rating denominator.


In [ ]:
table = counts.loc[counts.question_id == 28].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[28] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
values = np.repeat(table.numeric_value.to_numpy(), table.n.to_numpy())
assert len(values) == summary["valid_n"]
summary.update(median=float(np.median(values)), minimum=float(values.min()), maximum=float(values.max()))
display(pd.DataFrame([{k: summary[k] for k in ["median", "minimum", "maximum"]}]))
summaries[28] = summary
plot_question(table)


## Q29. Which strategic view best describes your organization?

In [ ]:
table = counts.loc[counts.question_id == 29].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[29] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
assert table.n.sum() == summary["valid_n"]
summaries[29] = summary
plot_question(table)


## Q30. Please describe the impact on your organization’s product lifecycle or strategic alignment: (e.g., lifecycle acceleration, innovation enablement, resource planning, scalability, long-term QA, budgeting, strategic fit.)

Lifecycle acceleration or time savings was coded in 6/9 responses. The question supplied examples of potential impacts, so these responses include endorsement of suggested impacts as well as respondents’ own descriptions.


In [ ]:
table = counts.loc[counts.question_id == 30].sort_values("category_order").copy()
table["percent"] = np.nan  # Post-submission reporting uses counts and denominators only.
table["n/N"] = [f"n = {n}/{valid}" for n, valid in zip(table.n, table.valid_n)]
show_table(table)
results[30] = table
summary = table[["question_id", "question_type", *DENOMINATORS]].iloc[0].to_dict()
summaries[30] = summary
plot_question(table)


## Coding coverage and result tables

Coverage includes observed responses receiving zero final codes. It excludes missing responses.

In [ ]:
for qid, distribution in coverage.groupby("question_id"):
    values = np.repeat(distribution.codes_per_response.to_numpy(), distribution.n_responses.to_numpy())
    assert len(values) == summaries[qid]["valid_n"]
    assert values.sum() == results[qid].n.sum()
    summaries[qid].update(no_code_n=int((values == 0).sum()), mean_codes=float(values.mean()),
                         min_codes=int(values.min()), max_codes=int(values.max()))
summary_table = pd.DataFrame(summaries.values()).sort_values("question_id")
display(summary_table.loc[summary_table.question_type == "coded", ["question_id", "valid_n", "no_code_n", "mean_codes", "min_codes", "max_codes"]])
pd.concat(results.values()).to_csv(OUTPUT / f"{SURVEY}_results.csv", index=False)
summary_table.to_csv(OUTPUT / f"{SURVEY}_summary.csv", index=False)
